# ch05 Bonus 01：在更大语料上预训练

> 对照官方 `ch05/03_bonus_pretraining_on_gutenberg`

## 一句话

主线用 2 万字的 `the-verdict.txt` demo。本 bonus 改用 **Project Gutenberg** 的更大语料继续预训练，观察损失降得更低、生成更连贯。

## 原理

数据规模是预训练效果的决定性因素之一。同样模型、同样训练步数，喂更多数据 → 损失更低 → 生成更连贯。本 notebook 展示「换更大语料」这条最直接的提升路径。

In [ ]:
from pathlib import Path
import urllib.request
import tiktoken

# Project Gutenberg 上的一篇较长文本（公版书《爱丽丝梦游仙境》）
GUTENBERG_URL = "https://www.gutenberg.org/files/11/11-0.txt"
gutenberg_path = Path("data/alice.txt")

# 如果本地没有，下载（约 17 万字符，是 the-verdict 的 ~8 倍）
if not gutenberg_path.exists():
    print("下载 Project Gutenberg 语料...")
    try:
        urllib.request.urlretrieve(GUTENBERG_URL, gutenberg_path)
        print("下载完成")
    except Exception as e:
        print(f"下载失败（离线/网络受限）: {e}")
        print("将回退到 the-verdict.txt 继续演示流程。")

# 统计规模对比
verdict_path = Path("data/the-verdict.txt")
v_size = len(verdict_path.read_text(encoding="utf-8"))
if gutenberg_path.exists():
    g_size = len(gutenberg_path.read_text(encoding="utf-8"))
    print(f"the-verdict: {v_size:,} 字符")
    print(f"alice(Gutenberg): {g_size:,} 字符  ← {g_size/v_size:.1f}× 更多")
else:
    print(f"the-verdict: {v_size:,} 字符（无 Gutenberg 数据）")

In [ ]:
# 用更大的语料继续训练主线模型，对比损失
import torch
from src.gpt import GPTModel, GPT_CONFIG_124M, create_dataloader_v1

data_path = gutenberg_path if gutenberg_path.exists() else verdict_path
with open(data_path, encoding="utf-8") as f:
    text = f.read()

# 小配置 demo（完整 124M 太慢）
cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4, "context_length": 256})

dl = create_dataloader_v1(text, batch_size=2, max_length=cfg["context_length"],
                          stride=cfg["context_length"], shuffle=True, drop_last=True)
print(f"语料 {len(text):,} 字符 → {len(dl)} 批次")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(123)
model = GPTModel(cfg).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)
model.train()

import torch.nn.functional as F
for epoch in range(3):
    total = 0; n = 0
    for x, y in dl:
        opt.zero_grad()
        loss = F.cross_entropy(model(x.to(device)).flatten(0,1), y.to(device).flatten())
        loss.backward(); opt.step()
        total += loss.item(); n += 1
    print(f"epoch {epoch}: loss {total/n:.4f}")
print("\n💡 数据更多 → loss 有更多下降空间，生成更连贯。")